# NB02 — Stage A: classical CNNs


## What this notebook runs

| | |
|---|---|
| **Architectures** | 4 |
| **Folds x seeds each** | 3 x 3 = 9 |
| **Total training runs** | **36** |
| **Estimated GPU time** | **~24 GPU-hours** |

### Wall-clock, by how many Kaggle accounts you run

| NUM_WORKERS | Wall-clock | Kaggle sessions each |
|---|---|---|
| 1 | ~23.6 h | 3 |
| 2 | ~11.9 h | 2 |
| 4 | ~6.1 h | 1 |

### Per architecture

| Architecture | Res | Batch | Epochs | Est. per run | x 9 runs |
|---|---|---|---|---|---|
| `resnet50` | 384 | 32 | 60 | 27 min | 4.0 h |
| `resnext50` | 384 | 32 | 60 | 32 min | 4.8 h |
| `densenet121` | 384 | 32 | 60 | 37 min | 5.5 h |
| `vgg16bn` | 384 | 16 | 60 | 61 min | 9.2 h |

> Estimates come from a **static** cost table calibrated against measured T4
> throughput. It stays static on purpose: if measurements fed back into the
> work split, two workers planning at different times would disagree about
> what they own, and a job gets trained twice while another is abandoned.

> **No early stopping.** Every run trains the full 60-epoch budget. Equal
> budget for every architecture is what keeps the comparison fair, and it means
> a run's length is known in advance -- which is what makes the estimate above
> honest.

> A Kaggle session lasts ~9-12 h and this pipeline pauses cleanly at 8.5 h, so a
> run needing more than one session resumes automatically. Just start a fresh
> session and re-run the notebook.



## Why this group is its own notebook

The field's reference points, and Grad-CAM's native home. They are cheap, they are what every prior tyre paper used, and VGG-16 in particular is where Grad-CAM was defined — which makes it a useful anchor for the XAI comparison in Stage E even though its accuracy will not lead.

Splitting Stage A by architecture family keeps each notebook inside one or two
Kaggle sessions, and means a failure in one family does not block the others.
All notebooks share the same library, the same registry and the same recipe —
so results across them are directly comparable.

> **The recipe is FIXED across the whole of Stage A.** Resolution, batch size,
> head, optimiser, schedule, sampler, epoch budget — all identical. If the
> recipe changes mid-sweep the architecture comparison stops being a
> comparison. Technique variation is Stage B's job.


In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow', 'timm'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
    'LyBzdGVwcykpCg==',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


## Step 1 — Session

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT   labels this Kaggle account in the shared run log. Two accounts
#           calling themselves the same thing makes the log useless.
# NUM_WORKERS  how many Kaggle accounts are running this notebook in parallel.
# WORKER_ID    0 .. NUM_WORKERS-1, DIFFERENT on each account.
#
# ---------------------------------------------------------------------------
# THESE TWO NUMBERS ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide only what this account STARTS FIRST. Whether a run is finished,
# and what epoch it reached, is read from HuggingFace -- from the run's own
# files -- so it is the same answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
ACCOUNT     = 'acct1'   # <<< CHANGE ME
NUM_WORKERS = 1         # <<< how many accounts are running in parallel
WORKER_ID   = 0         # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='a',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


## Step 2 — Dataset

In [ ]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


## Step 3 — Catch up with every account

In [ ]:
# === Catch up with what every account has already done =====================
# Two different questions, answered by two different sources.
#
#   registry   who CLAIMED what. Per-worker shard files, because HuggingFace
#              has no append and a shared ledger would silently lose writes.
#              Useful for one thing only: is somebody on this run right now.
#
#   inventory  what the repository actually HOLDS. runs/<id>/STATUS.json either
#              says epoch 34 or it does not, and it says the same thing to
#              every account at every value of NUM_WORKERS.
#
# Work planning reads the inventory. That is what makes the worker count safe
# to change.
sess.sync_state(verbose=True)
sess.push_now('sync complete')       # <- push at the end of an important cell


## Step 4 — See the plan and the time before committing to it

If the split looks badly unbalanced, or the hours are more than you want on one
account, change `NUM_WORKERS` **now** rather than discovering it on day three.

In [ ]:
ARCHS = ['resnet50', 'resnext50', 'densenet121', 'vgg16bn']
FOLDS = (0, 1, 2)
SEEDS = (1, 2, 3)

# ~15 s. Builds each architecture (no pretrained download) and forwards one
# batch at the resolution it will actually be fed. RAISES if any cannot.
#
# NB05 was launched on four accounts without this. All 18 dinov2 runs died on
# their first batch -- `vit_*_patch14_dinov2` is created at img_size=518 and
# its patch embedding asserts an exact match against the 392 we feed it. The
# check that would have caught it took fifteen seconds.
tl.assert_zoo_ok(ARCHS)

cfgs = sess.configs(ARCHS, FOLDS, SEEDS, technique='base')
run_ids = [c['run_id'] for c in cfgs]

est = tl.estimate_phase(run_ids, num_workers=NUM_WORKERS)
print(f"runs in this notebook : {est['n_runs']}")
print(f"total GPU time        : ~{est['total_gpu_hours']:.1f} GPU-hours")
print(f"at NUM_WORKERS={NUM_WORKERS:<2d}      : ~{est['wall_clock_hours']:.1f} h wall-clock "
      f"({est['sessions_needed']} Kaggle session(s) on this account)")
print()
for nw in (1, 2, 4):
    if nw > est['n_runs']:
        break
    e = tl.estimate_phase(run_ids, nw)
    print(f"  NUM_WORKERS={nw}: ~{e['wall_clock_hours']:5.1f} h wall-clock"
          + ('   <-- you' if nw == NUM_WORKERS else ''))
print()
rep = tl.shard_report(run_ids, max(NUM_WORKERS, 1))
print(rep.to_string(index=False))
print(f"imbalance: {rep.attrs.get('imbalance', 1.0)}x   (1.0 is perfect)")


## Step 4b — What is already on HuggingFace, and what this session will do

Read this table before every long run. It is the answer to the only question
that matters — *am I about to redo work that is already done* — taken from the
files rather than from anybody's bookkeeping.

`state` comes from the repository. `registry` comes from the run log. When they
disagree, **the repository is right**: a run the registry calls `failed`
because it hit an exception at epoch 47 still has a checkpoint at epoch 47, and
`action` will correctly say `resume`.

In [ ]:
recon = sess.reconcile(run_ids)

## Step 5 — Train

**Safe to stop at any moment.** SIGTERM, Ctrl-C, an uncaught exception and the
8.5-hour watchdog all trigger an immediate push before anything is lost. Start
a fresh session and re-run this notebook to continue exactly where it stopped.

Each epoch shows a live progress bar; the summary line after it carries
`val_QWK` (the ordinal metric we select on) and `dl` (the fraction of the epoch
spent waiting for data — if that is high the fix is the dataloader, not the
model).

HuggingFace receives: metrics every epoch, checkpoints every epoch, telemetry
every 10 epochs, and a **blocking push the moment each model finishes**.

In [ ]:
summaries = sess.run_all(cfgs, title='Stage A')

## Step 6 — Results, from HuggingFace, against the floor

`aggregate_remote()` — not `aggregate()`. The local one globs this session's
staging directory, so on four accounts each one produces a table of the eleven
runs it happened to do. Nobody ever sees all thirty-six, which is the only view
that answers anything.

Two numbers per model, and the gap between them is the point:

| | chosen by | honest? |
|---|---|---|
| `best_val_*` | the epoch with the highest val QWK | **no** — selected by looking at the 4-tyre validation fold |
| `final_val_*` | epoch 60, fixed budget | yes — nobody chose it |

In [ ]:
df = sess.aggregate_remote(run_ids)
if len(df):
    print()
    g = sess.honest_table(df)
    print('\n\nTrivial baselines (macro-F1 per fold):')
    print(tl.baseline_table().to_string(index=False))
else:
    print('nothing on HuggingFace yet for these run ids')


## Step 7 — Progress across ALL accounts

If `trained more than once` appears here, two accounts did the same run. That
is wasted GPU time, not a correctness problem — the results are identical.

In [ ]:
import pandas as pd
sess.sync_state(run_ids, verbose=False)
state = sess.registry.latest()
rows = [{'run_id': r, 'state': state.get(r, {}).get('state', 'not started'),
         'epoch': sess.inventory.epoch(r),
         'qwk': sess.inventory.qwk(r),
         'by': state.get(r, {}).get('account')} for r in sorted(run_ids)]
prog = pd.DataFrame(rows)
print(prog.to_string(index=False))
n_done = int((prog.state == 'completed').sum())
print(f'\n{n_done} of {len(prog)} runs finished across all accounts '
      f'({n_done/max(1,len(prog))*100:.0f}%)')


## Step 8 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# Draining the upload queue is NOT the same as the files being on HuggingFace.
# Ask the repository before you close this tab.
#
# Three states, not two. FINISHED and RESUMABLE are both safe -- a run paused
# at epoch 34 whose ckpt_last.pt is on HF loses nothing when you close the tab.
# Only AT RISK (no summary.json AND no checkpoint) needs action.
sess.confirm_on_hf(run_ids)
